# 🧬 ⚙️ CpGPT Array Conversion Tutorial ⚙️ 🧬

Welcome to the CpGPT Array Conversion Tutorial! 👋 

In this notebook, we'll walk you through the process of converting between different array formats.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Load CpGPT Dependencies](#2-load-cpgpt-dependencies)
3. [Load Pre-trained Model](#3-load-pre-trained-model)
4. [Save Original Data](#4-save-original-data)
5. [Create Dataset](#5-create-dataset)
6. [Convert Illumina Probes](#6-convert-illumina-probes)
7. [Generate DNA LLM Embeddings](#7-generate-dna-llm-embeddings)
8. [Embed and Reconstruct Methylome](#8-embed-and-reconstruct-methylome)
9. [Visualize Reconstructed Methylome](#9-visualize-reconstructed-methylome)
10. [Next Steps](#10-next-steps)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements. The bulk of the variables are defined here below. The pre-trained model checkpoint alongside DNA embedding files can also be downloaded if not present.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The download cell below downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

### 1.1 Download dependencies and define variables

In [ ]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="small", species="human")

In [ ]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
PROCESSED_DIR = "../data/tutorials/processed/convert_array"
ARROW_DF_PATH = "../data/tutorials/raw/toy.arrow"
ARROW_DF_SUBSET_PATH = "../data/tutorials/raw/toy_subset.arrow"

# The maximum context length to give to the model
MAX_INPUT_LENGTH = 10_000

# Device configuration
DEVICE = "cuda"

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

> **⚠️ Warning**
> 
> It is recommended to have a GPU for inference as CPU might be slow.
> 
> Reconstructing the methylome for a few hundred samples might take up to one hour on a CPU. ⌛
>
> This might be a great exercise in testing your patience.

### 1.2 Import packages


In [ ]:
# Standard library imports
import os
import shutil
import warnings

import requests

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use(
    [
        "science",
        "nature",
    ],
)
import pyaging as pya
import seaborn as sns

# Lightning imports
from lightning.pytorch import seed_everything
from scipy.stats import pearsonr, spearmanr

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.components.cpgpt_dataset import CpGPTDataset
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.utils.rich_utils import create_rich_dataset_preview

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

## 2. Load CpGPT Dependencies

Now that we have our basic environment set up, we need to load three crucial components for CpGPT:

1. **DNA LLM Embedder**: This component contains the embeddings for all CpGs of interest. It's essential for converting DNA sequences into a format that our model can understand and process.

2. **Illumina Methylation Prober**: This tool allows us to convert probe names to genomic locations, which is required for working with Illumina methylation array data.

3. **CpG Inferencer**: This component is responsible for loading the pre-trained CpGPT model checkpoint and any additional configurations or overrides needed for the fine-tuning process.

### 3.1 Load DNALLMEmbedder

In [ ]:
embedder = DNALLMEmbedder(dependencies_dir=DEPENDENCIES_DIR)

### 3.2 Load IlluminaMethylationProber

In [ ]:
prober = IlluminaMethylationProber(dependencies_dir=DEPENDENCIES_DIR, embedder=embedder)

### 3.3 Load CpGInferencer

In [ ]:
inferencer = CpGPTInferencer()

## 3. Load Pre-trained Model

We need to load a pre-trained CpGPT model to be able to reconstruct the beta values for different CpG sites. Generally, the pre-trained model works well for zero-shot imputation but you might want to finetune the model using data that is more appropriate for your imputation goals. For instance, finetuning on RRBS data before following this tutorial.

In [ ]:
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)
model_info = {"current_hparams": config}
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

## 4. Save Original Data

We'll prepare our methylation data for reconstruction and save it in a format that CpGPT can efficiently process. We first begin with a dataset that is already stored in .arrow (feather v2) format. Then, let's create a memory-mapped dataset.

For demonstration purposes, let's imagine the completely hypothetical scenario where some big manufacturer of methylation arrays decides to update their platform by removing half of the probes without any prior notice. Again, hypothetical but if it does happen CpGPT has you covered.

In [ ]:
# Read the arrow file
df = pd.read_feather(ARROW_DF_PATH)

# Filter df to only include CpGs from the Illumina Prober
df = df.loc[
    :,
    np.intersect1d(df.columns, list(prober.illumina_metadata_dict["homo_sapiens"].keys())),
]

# Show first few rows
df.head()

In [ ]:
# Download the MSA manifest file
url = "https://github.com/zhou-lab/InfiniumAnnotationV1/raw/main/Anno/MSA/MSA.hg38.manifest.tsv.gz"
local_filename = os.path.join(PROCESSED_DIR, "MSA.hg38.manifest.tsv.gz")

Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

response = requests.get(url, stream=True)
with open(local_filename, "wb") as f:
    shutil.copyfileobj(response.raw, f)

# Read the downloaded MSA manifest file
msa_manifest_df = pd.read_table(local_filename, sep="\t")

# Extract the Probe_ID without the suffix (if any)
msa_manifest_df["Probe_ID"] = msa_manifest_df["Probe_ID"].apply(lambda x: x.split("_")[0])

# Get unique probe IDs
msa_probes = np.unique(msa_manifest_df["Probe_ID"].tolist())

# Show first few probes
msa_probes[0:5]

In [ ]:
# Get only the probes that are also in the MSA manifest
df_subset = df.loc[:, np.intersect1d(df.columns, msa_probes)]

# Save subset df
df_subset.to_feather(ARROW_DF_SUBSET_PATH)

# Show first few rows
df_subset.head()

In [ ]:
# Define datasaver
convert_array_datasaver = CpGPTDataSaver(
    data_paths=ARROW_DF_SUBSET_PATH,
    processed_dir=PROCESSED_DIR,
)

# Process the file
convert_array_datasaver.process_files(prober, embedder)

## 5. Create Dataset

We need to define a way that the CpGPT model can efficiently access our memory-mapped dataset. Let's define a CpGPTDataset object and check the output.

In [ ]:
convert_array_data = CpGPTDataset(
    embedder,
    processed_dir=PROCESSED_DIR,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy=model_info["current_hparams"]["data"]["sorting_strategy"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

Let's double check the outputs make sense:

In [ ]:
create_rich_dataset_preview(convert_array_data, "Convert Array Data Sample")

## 6. Convert Illumina Probes

In order for the model to understand what part of the methylome needs to be reconstructed, a genomic location is necessary. This can be given directly in the format '12:213103' or first converted from the probes of the Illumina methylation arrays. The IlluminaMethylationProber class has a handy function for that conversion for the most up-to-date arrays as of the publication of this tutorial.

In [ ]:
# Get probes that were removed from the array
probes = [col for col in df.columns if col not in df_subset.columns]

# Convert probes to genomic locations
genomic_locations = prober.locate_probes(probes, "homo_sapiens")

# Show first few locations
genomic_locations[0:5]

## 7. Generate DNA LLM Embeddings
 
In this section, we'll generate any DNA embeddings for the genomic locations of our dataset and for our genomic locations of interest that might be missing. The model requires that all genomic locations have embeddings from a DNA LLM or it will fail otherwise. The easiest way is to download such embeddings if you are working with human samples ([step 1.3](#1.3-download-dependencies)) and Illumina arrays or it might take a little bit for the processing to happen.

In [ ]:
# Embed genomic locations from the subset data
all_species = list(convert_array_datasaver.all_genomic_locations.keys())

# Loop through each species
for species in all_species:
    dataset_genomic_locations = list(
        convert_array_datasaver.all_genomic_locations.get(species, []),
    )

    embedder.parse_dna_embeddings(
        dataset_genomic_locations,
        species,
        dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
        dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    )

In [ ]:
# Embed genomic locations of interest from the removed probes
embedder.parse_dna_embeddings(
    genomic_locations,
    "homo_sapiens",  # The imputation is done for a single species
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
)

## 8. Embed and Reconstruct Methylome

First, let's use CpGPT to create a sample embedding given the available information in the CpG sites. Then, let's ask it to reconstruct the beta values for the genomic locations of interest. We also get the uncertainty for each reconstruction from the model.

### 8.1 Reconstruct the whole methylome

In [ ]:
# Embed samples
sample_embeddings = inferencer.embed_sample(model, convert_array_data, batch_size=4)

In [ ]:
# Reconstruct beta values
X_converted, X_unc = inferencer.reconstruct_betas(
    model,
    sample_embedding=sample_embeddings,
    genomic_locations=genomic_locations,
    embedder=embedder,
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    species="homo_sapiens",  # You need to pick a single species
    batch_size=1,
    return_type="numpy",
)

In [ ]:
# Transform to DataFrame
X_converted_df = pd.DataFrame(X_converted, columns=probes, index=df.index)
X_unc_df = pd.DataFrame(X_unc, columns=probes, index=df.index)

# Expand the subset data with the reconstructed data
df_converted = pd.concat([df_subset, X_converted_df], axis=1)

## 9. Visualize Reconstructed Methylome

As a quick sanity check, let's create some plots to try to understand what is going on. We'll visualize:

1. The bimodal distribution of imputed beta values
2. The distribution of uncertainty with higher uncertainty for values close to 0.5
3. The predictions of epigenetic clocks from the reconstructed values

These visualizations will help us ensure that our model is working as expected. If not, it might be worth finetuning with the appropriate data.

### 9.1 Distribution of reconstructed beta values

In [ ]:
# Create a histogram for the distribution of beta values for converted and original values

# Set up the matplotlib figure
fig, ax = plt.subplots(figsize=(6, 4))

# Select 2 samples randomly
sample_size = 2
sample_indices = np.random.choice(X_converted_df.index, size=sample_size, replace=False)

# Filter the dataframes to include only the selected samples
df_converted_sample = X_converted_df.loc[sample_indices]
df_original = df.loc[sample_indices]

# Melt the dataframes to long format for easier plotting
df_converted_long = df_converted_sample.melt(var_name="CpG Site", value_name="Beta Value")
df_converted_long["Type"] = "Converted"
df_original_long = df_original.melt(var_name="CpG Site", value_name="Beta Value")
df_original_long["Type"] = "Original"

# Combine the melted dataframes
df_combined = pd.concat([df_converted_long, df_original_long])

# Create the histogram and KDE plot
sns.histplot(
    data=df_combined,
    x="Beta Value",
    hue="Type",
    kde=True,
    stat="density",
    bins=50,
    alpha=0.5,
    ax=ax,
)

# Calculate mean and median of converted and original values
mean_converted = df_converted_long["Beta Value"].mean()
median_converted = df_converted_long["Beta Value"].median()
mean_original = df_original_long["Beta Value"].mean()
median_original = df_original_long["Beta Value"].median()

# Add mean and median lines
ax.axvline(
    mean_converted,
    color="red",
    linestyle="dashed",
    linewidth=2,
    label=f"Converted Mean: {mean_converted:.3f}",
)
ax.axvline(
    median_converted,
    color="darkred",
    linestyle="dashed",
    linewidth=2,
    label=f"Converted Median: {median_converted:.3f}",
)
ax.axvline(
    mean_original,
    color="blue",
    linestyle="dashed",
    linewidth=2,
    label=f"Ground Truth Mean: {mean_original:.3f}",
)
ax.axvline(
    median_original,
    color="darkblue",
    linestyle="dashed",
    linewidth=2,
    label=f"Ground Truth Median: {median_original:.3f}",
)

# Customize the plot
ax.set_title("Distribution of Converted and Ground Truth Beta Values\n(3 Samples)")
ax.set_xlabel("Beta Value")
ax.set_ylabel("Density")
ax.set_xlim(0, 1)
ax.legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

### 9.2 Uncertainty of reconstructed beta values

In [ ]:
# Create a scatter plot with uncertainty using matplotlib and seaborn, only for masked values

# Summarize X_imp_masked and X_unc_masked by feature
X_imp_summary = X_converted_df.mean().reset_index()
X_imp_summary.columns = ["Feature", "Mean Imputed Value"]

X_unc_summary = X_unc_df.mean().reset_index()
X_unc_summary.columns = ["Feature", "Mean Uncertainty"]

# Calculate confidence interval using the distribution of X_unc_masked
X_unc_quantiles = X_unc_df.quantile([0.025, 0.975])
X_unc_lower = X_unc_quantiles.iloc[0].reset_index()
X_unc_upper = X_unc_quantiles.iloc[1].reset_index()
X_unc_lower.columns = ["Feature", "CI_lower"]
X_unc_upper.columns = ["Feature", "CI_upper"]

# Merge the summaries and confidence intervals
merged_summary = pd.merge(X_imp_summary, X_unc_summary, on="Feature")
merged_summary = pd.merge(merged_summary, X_unc_lower, on="Feature")
merged_summary = pd.merge(merged_summary, X_unc_upper, on="Feature")

# Sort by Mean Imputed Value for better visualization
merged_summary = merged_summary.sort_values("Mean Imputed Value")

# Create the scatter plot with uncertainty
plt.figure(figsize=(5, 3))

# Plot the main scatter points
scatter = plt.scatter(
    merged_summary["Mean Imputed Value"],
    merged_summary["Mean Uncertainty"],
    c=merged_summary["Mean Uncertainty"],
    cmap="viridis",
    alpha=0.4,
    s=10,
)

# Add error bars for uncertainty, ensuring non-negative values
yerr_lower = np.maximum(merged_summary["Mean Uncertainty"] - merged_summary["CI_lower"], 0)
yerr_upper = np.maximum(merged_summary["CI_upper"] - merged_summary["Mean Uncertainty"], 0)

plt.errorbar(
    merged_summary["Mean Imputed Value"],
    merged_summary["Mean Uncertainty"],
    yerr=[yerr_lower, yerr_upper],
    fmt="none",
    ecolor="lightgray",
    alpha=0.3,
)

# Customize the plot
plt.title("Uncertainty of Reconstructed Beta Values")
plt.xlabel("Mean Reconstructed Beta Value")
plt.ylabel("Mean Uncertainty (with 95% CI)")
plt.xlim(0, 1)  # Beta values are between 0 and 1
plt.colorbar(scatter, label="Mean Uncertainty")

# Add grid
plt.grid(True, linestyle="--", alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()

### 9.3 Epigenetic clock prediction

In [ ]:
# Define clocks of interest from pyaging
clocks = [
    "horvath2013",
    "dnamphenoage",
    "pcdnamtl",
    "grimage2",
    "epitoc1",
    "hannum",
    "skinandblood",
    "pcphenoage",
    "intrinclock",
    "stemtoc",
    "han",
    "knight",
    "leerefinedrobust",
    "stocz",
    "dnamtl",
    "encen40",
]

# Predict age using original beta values
adata_original = pya.pp.df_to_adata(df, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_original, clocks, verbose=False)

# Predict age using subset beta values
adata_subset = pya.pp.df_to_adata(df_subset, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_subset, clocks, verbose=False)

# Predict age using imputed beta values for each method
adata_converted = pya.pp.df_to_adata(df_converted, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_converted, clocks, verbose=False)

# Calculate correlations and MAE for each clock and imputation method
results = {}

# Set up the subplot grid
fig, axs = plt.subplots(len(clocks) // 2, 2, figsize=(10, 20))
fig.suptitle("Epigenetic Clock Predictions", fontsize=16)

for i, clock in enumerate(clocks):
    original = np.array(adata_original.obs[clock])
    converted = np.array(adata_converted.obs[clock])
    subset = np.array(adata_subset.obs[clock])

    row = i // 2
    col = i % 2

    # Create scatter plot for each method
    axs[row, col].scatter(original, subset, label="Original", alpha=0.6, color="blue")
    axs[row, col].scatter(original, converted, label="CpGPT", alpha=0.6, color="red")

    # Add identity line
    min_val = min(original.min(), converted.min(), subset.min())
    max_val = max(original.max(), converted.max(), subset.max())
    axs[row, col].plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.7)

    # Add regression lines and dashed lines
    for method, data, color in [("Original", subset, "blue"), ("CpGPT", converted, "red")]:
        # Regression line
        z = np.polyfit(original, data, 1)
        p = np.poly1d(z)
        # Dashed line
        axs[row, col].plot(
            original,
            p(original),
            color=color,
            linestyle="dashed",
            linewidth=0.3,
            alpha=0.5,
        )

    axs[row, col].set_xlabel("Ground Truth")
    axs[row, col].set_ylabel("Imputed/Subset Prediction")
    axs[row, col].set_title(clock)
    axs[row, col].legend()

    # Calculate and add text annotation with correlations and MAE
    for method, data in [("Original", subset), ("CpGPT", converted)]:
        pearson_corr, _ = pearsonr(original, data)
        spearman_corr, _ = spearmanr(original, data)
        mae = np.mean(np.abs(original - data))

        if clock not in results:
            results[clock] = {}
        results[clock][method] = {"Pearson": pearson_corr, "Spearman": spearman_corr, "MAE": mae}

        y_pos = 0.95 if method == "Original" else 0.75  # Adjust vertical position
        axs[row, col].text(
            0.05,
            y_pos,
            f"{method}:\nPearson: {pearson_corr:.4f}\nSpearman: {spearman_corr:.4f}\nMAE: {mae:.4f}",
            transform=axs[row, col].transAxes,
            verticalalignment="top",
            bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": "black", "alpha": 0.7},
        )

# Adjust layout
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

In [ ]:
# Calculate correlations and normalized MAE for each clock and imputation method
results = {}

for clock in clocks:
    original = np.array(adata_original.obs[clock])
    converted = np.array(adata_converted.obs[clock])
    subset = np.array(adata_subset.obs[clock])

    # Calculate min and max of ground truth for normalization
    min_val = np.min(original)
    max_val = np.max(original)
    range_val = max_val - min_val

    results[clock] = {}
    for method, data in [("Original", subset), ("CpGPT", converted)]:
        pearson_corr, _ = pearsonr(original, data)
        spearman_corr, _ = spearmanr(original, data)
        # Calculate normalized MAE
        normalized_mae = np.mean(np.abs(original - data) / range_val)

        results[clock][method] = {
            "Pearson": pearson_corr,
            "Spearman": spearman_corr,
            "Normalized MAE": normalized_mae,
        }

# Prepare data for heatmaps
methods = ["Original", "CpGPT"]
metrics = ["Spearman", "Pearson", "Normalized MAE"]

# Create a figure with 3 subplots
fig, axs = plt.subplots(1, 3, figsize=(7, 6))

for idx, metric in enumerate(metrics):
    heatmap_data = []
    for clock in clocks:
        row = [results[clock][method][metric] for method in methods]
        heatmap_data.append(row)

    # Convert to numpy array
    heatmap_data = np.array(heatmap_data)

    # Calculate robust min and max values (e.g., 5th and 95th percentiles)
    vmin, vmax = np.percentile(heatmap_data, [10, 90])

    # Create heatmap with robust color scaling
    im = axs[idx].imshow(heatmap_data, vmin=vmin, vmax=vmax, cmap="YlOrBr")

    # Set tick labels
    axs[idx].set_xticks(np.arange(len(methods)))
    axs[idx].set_yticks(np.arange(len(clocks)))
    axs[idx].set_xticklabels(methods)
    axs[idx].set_yticklabels(clocks)

    # Rotate the tick labels and set their alignment
    plt.setp(axs[idx].get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add colorbar
    cbar = axs[idx].figure.colorbar(im, ax=axs[idx], extend="both")
    cbar.ax.set_ylabel(
        f"{metric} {'Correlation' if 'MAE' not in metric else ''}",
        rotation=-90,
        va="bottom",
    )

    # Add text annotations
    for i in range(len(clocks)):
        for j in range(len(methods)):
            text = axs[idx].text(
                j,
                i,
                f"{heatmap_data[i][j]:.2f}",
                ha="center",
                va="center",
                color="black",
            )

    axs[idx].set_title(f"Epigenetic Clock Predictions - {metric}")

fig.tight_layout()
plt.show()

## 10. Next steps

Here are some ideas for further exploration:

- Experiment with different input context lengths
- Use various pre-trained CpGPT models and compare their array conversion for different tissues